# Back to the feature: an end-to-end paper recommender

_Feature Engineering (Zheng & Casari)_

**Compose the book's transforms into one pipeline — one-hot the categories, scale the numbers, normalize, and retrieve nearest neighbors.**

A content recommender is just "turn each item into a vector, then find the nearest vectors". All the work is in turning the item into a good vector — that is feature engineering — and in making the geometry of "nearest" meaningful — that is scaling and normalization.

       For an academic paper, the vector is built by concatenating the outputs of simple transforms: a one-hot / bag-of-words block for the fields of study, plus a handful of scaled numeric columns (year, citation count). Stack these side by side and you get one feature row per paper, i.e. a feature matrix $X$ with one row per item.

---

This notebook is a practice scaffold for the **Back to the feature: an end-to-end paper recommender** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix

# Microsoft Academic Graph papers, prepared as in Chapter 9.
# Data: github.com/alicezheng/feature-engineering-book (Chapter 9).
# Each row is a paper with its fields of study and a few numeric features.
papers = pd.read_csv('mag_papers.csv')
# columns: 'paper_id', 'title', 'fields_of_study' (space-joined string),
#          'year', 'citation_count'

# ---- ROUND 1: fields of study only (one-hot / bag-of-words) ----------
# Each paper's fields are a bag of category tokens; one-hot encode them.
fields_vec = CountVectorizer(binary=True, token_pattern=r'[^ ]+')
X_fields = fields_vec.fit_transform(papers['fields_of_study'])   # sparse 0/1

def build_recommender(X, k=5):
    """L2-normalize rows, then fit a cosine nearest-neighbor index."""
    Xn = normalize(X, norm='l2', axis=1)          # unit-length rows
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine')
    nn.fit(Xn)
    return nn, Xn

def recommend(nn, Xn, query_idx, titles, k=5):
    dist, idx = nn.kneighbors(Xn[query_idx], n_neighbors=k + 1)
    # neighbor 0 is the query itself; drop it
    for d, j in zip(dist[0][1:], idx[0][1:]):
        print(f"  cos={1 - d:.3f}  {titles.iloc[j]}")

nn1, Xn1 = build_recommender(X_fields)
print("ROUND 1 (fields only):")
recommend(nn1, Xn1, query_idx=0, titles=papers['title'])

# ---- ROUND 2: add SCALED numeric features, then re-measure -----------
# Raw year/citations live on a totally different scale than the 0/1 flags,
# so we standardize them before appending. This is the key iterative step.
num = papers[['year', 'citation_count']].to_numpy(dtype=float)
num_scaled = StandardScaler().fit_transform(num)          # mean 0, var 1
X_full = hstack([X_fields, csr_matrix(num_scaled)]).tocsr()

nn2, Xn2 = build_recommender(X_full)
print("ROUND 2 (fields + SCALED year & citations):")
recommend(nn2, Xn2, query_idx=0, titles=papers['title'])

# Feature engineering is ITERATIVE: add a feature, rescale, re-measure.
# Compare the two rounds' neighbors by hand and keep what improves them.
# (Optional next rounds: TfidfTransformer() to weight rare fields,
#  or TruncatedSVD/PCA to compress the wide sparse one-hot matrix.)

## Visualize the data & results

_On a bundled item catalog, does scaling + L2-normalizing the features before cosine nearest-neighbor retrieval return more same-class items (higher precision@k) than using RAW features?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors

# Bundled dataset as a stand-in ITEM CATALOG: 178 wines, 13 features, 3 classes.
X, y = load_wine(return_X_y=True)
k = 5

def precision_at_k(features, labels, k):
    """For each item, retrieve k nearest neighbors by cosine similarity;
    measure the fraction that share the item's class (same-class retrieval)."""
    Xn = normalize(features, norm='l2', axis=1)        # unit-length rows
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(Xn)
    _, idx = nn.kneighbors(Xn)
    hits = []
    for i in range(len(labels)):
        neigh = idx[i][1:]                              # drop the item itself
        hits.append(np.mean(labels[neigh] == labels[i]))
    return float(np.mean(hits))

# RAW: features on wildly different scales (e.g. 'proline' is in the hundreds).
raw_p = precision_at_k(X, y, k)

# SCALED + normalized: standardize every column first, then L2-normalize rows.
X_scaled = StandardScaler().fit_transform(X)
scaled_p = precision_at_k(X_scaled, y, k)

print([round(raw_p, 2), round(scaled_p, 2)])
# -> [0.76, 0.93]   scaling before a distance-based retrieval helps a lot

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You one-hot encode 5000 fields of study and append one raw "citation count" column (values up to ~50000), then retrieve nearest neighbors by cosine similarity. Every recommendation is a hugely-cited paper on an unrelated topic. What went wrong and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the magnitude of the citation column to the 0/1 field columns. — _The field flags are 0 or 1; the citation count is in the tens of thousands, so its square swamps the sum of squares in the vector length._
- Recognize the distance is now dominated by one feature. — _Cosine similarity is driven almost entirely by the citation column, so papers look similar when their citation counts are similar, regardless of topic._
- Scale every column before measuring distance. — _Standardizing (or min-max scaling) the citation column to the same order of magnitude as the 0/1 flags lets the fields of study contribute to the similarity again._

**Answer:** The unscaled citation column dominates the distance: with values up to 50000 its square dwarfs the 0/1 field indicators, so cosine similarity ranks papers by citation magnitude rather than topic. Fix: scale/normalize every feature first — standardize the citation column to mean 0, variance 1 (or min-max to $[0,1]$) so it sits on the same scale as the field flags — then L2-normalize each row and re-run nearest neighbors. Now the shared fields of study drive the recommendations.

</details>

**Problem 2.** Your first recommender uses only fields of study and gives mediocre results. A teammate says "just switch from cosine to Euclidean distance and it'll be fixed." Per the book's Chapter 9 philosophy, is that the right move?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what actually drives a content recommender's quality. — _The book stresses that the features and their scaling matter far more than the choice of similarity metric._
- Think about the iterative feature-engineering loop. — _Chapter 9 improves the recommender by adding raw features (year, citations) and re-scaling them, then re-inspecting neighbors — not by swapping metrics._
- Decide the next action. — _Adding and scaling informative features addresses the root cause; changing the metric on the same weak feature set rarely helps much._

**Answer:** No — swapping the metric is the wrong lever. The book's lesson is that feature engineering is iterative and that features and their scaling drive quality far more than the distance metric. The right move is to add more raw features (publication year, citation/reference counts), scale and normalize them so none dominates, re-run nearest neighbors, and inspect the recommendations. If that still under-performs, consider tf-idf weighting of the fields or PCA to compress the wide one-hot matrix. Iterate: add a feature, rescale, re-measure.

</details>